# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

### Unit of Analysis
- **One row = one content item** (`content_id`) for a specific client (`client_id`).
- Every page has a unique pseudonymous identifier `content_id` that is unique across the entire dataset. This makes a single content page the fundamental grain of our analysis.

### Time Window
- **Trailing 90 days**: All search performance (impressions, clicks, avg_position) and user engagement metrics (pageviews, sessions, users, scroll events, engagement sessions, AI traffic) are aggregated over a fixed trailing 90-day window ending at export time.
- **30-day Comparison sub-windows**: To measure trend directions and labels, the metrics are also broken down into two sub-windows:
  - `last_30d`: The most recent 30 days.
  - `prev_30d`: The preceding 30 days (days 31-60 back).
- **Age and freshness**: The dataset represents a snapshot where every page has a history of at least 90 days (`content_age_days >= 90`) and has at least one search impression in the last 90 days (`impressions_90d >= 1`).

In [ ]:
import pandas as pd

# Load starter dataset
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

# 1. Verify Unit of Analysis / Grain uniqueness
total_rows = len(df)
unique_pages = df['content_id'].nunique()
unique_clients = df['client_id'].nunique()

print(f"Total rows in dataset: {total_rows:,}")
print(f"Unique content_id values (pages): {unique_pages:,}")
print(f"Unique client_id values (clients): {unique_clients:,}")
print(f"Is content_id unique? {unique_pages == total_rows}")

# 2. Verify Time Window restrictions
min_age = df['content_age_days'].min()
max_age = df['content_age_days'].max()
min_impressions = df['impressions_90d'].min()

print(f"Page age range (days): {min_age} to {max_age}")
print(f"Minimum GSC impressions over 90 days: {min_impressions}")

## 2. Fields: feature / label / context / excluded

We classify every field in the 44-column starter dataset into exactly one of the four categories to establish a clean contract and prevent leakage:

### 1. Feature (Safe to use, knowable BEFORE the prediction)
These are search metrics, page properties, and keyword context signals:
- **Keyword Context**: `search_volume`, `competition`, `competition_level`, `cpc`, `content_type`, `main_intent`
- **Content Properties**: `word_count`, `char_count`, `content_age_days`, `days_since_last_update`, `freshness_tier`, `word_count_tier`, `char_count_tier`
- **90-Day Performance Totals**: `impressions_90d`, `clicks_90d`, `pageviews_90d`, `sessions_90d`, `users_90d`, `engaged_sessions_90d`, `ai_sessions_90d`, `scroll_events_90d`, `days_with_impressions`, `days_with_sessions`
- **Derived Rates**: `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`
- **Tiers/Buckets**: `age_tier`, `age_tier_order`, `impression_tier`, `position_tier`
- **Comparison window (prev 30d)**: `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d` (Note: for the starter CSV, the trend label is computed from last 30d vs prev 30d, but for a true production model we'd use prev 30d or older to predict future decline).

### 2. Label / Proxy (The thing we predict)
- `trend_direction` (e.g. `down` indicates declining content, which maps to `is_declining_label`).
- `trend_pct` (from which `trend_direction` is computed).
- *Never use these as features; doing so introduces 100% target leakage.*

### 3. Context (Grouping / splitting only, never features)
- `content_id`: Unique page identifier.
- `client_id`: Identifies the client; must be used for client-holdout splits (grouped train/test) to ensure the model generalizes to new clients.

### 4. Excluded
- `provider_used` and `model_used`: Metadata indicating which LLM was used to generate the article. Excluded because they do not reflect search performance/engagement and could inject model-specific bias without a causal relationship.
- `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`: Excluded because they are used directly in calculating the trend label (`trend_pct` and `trend_direction`). Using them as features would introduce severe target leakage because they overlap with the label definition window.

In [ ]:
# Show columns mapped to each category
features = [
    'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent',
    'word_count', 'char_count', 'content_age_days', 'days_since_last_update', 'freshness_tier',
    'word_count_tier', 'char_count_tier', 'impressions_90d', 'clicks_90d', 'pageviews_90d',
    'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
    'days_with_impressions', 'days_with_sessions', 'ctr', 'avg_position', 'engagement_rate',
    'scroll_rate', 'ai_traffic_pct', 'age_tier', 'age_tier_order', 'impression_tier',
    'position_tier', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d'
]

labels = ['trend_direction', 'trend_pct']

context = ['content_id', 'client_id']

excluded = [
    'provider_used', 'model_used', 
    'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d'
]

print(f"Total columns in source: {len(df.columns)}")
print(f"Classified columns: {len(features) + len(labels) + len(context) + len(excluded)}")
assert len(df.columns) == len(features) + len(labels) + len(context) + len(excluded)

## 3. Verify it with queries (grain, counts, missing values, windows)

We verify the claims in the contract with concrete queries on the starter dataset:

1. **Grain**: Grouping by `content_id` and confirming no duplicate records exist.
2. **Counts**: Total rows, unique clients, and baseline rate of decline (`trend_direction == "down"`).
3. **Missingness**: Analyzing missing value percentages per column and checking if missingness follows a systematic pattern based on `content_type` (since a blind `fillna(0)` can inject a category signal).
4. **Windows**: Verifying `content_age_days` and update histories.

In [ ]:
import numpy as np

# 1. Grain Check
print("=== 1. Grain Uniqueness Check ===")
grain_dups = df.groupby('content_id').size()
dups_count = (grain_dups > 1).sum()
print(f"Number of content_id values with >1 row: {dups_count}")

# 2. Counts and Base Rate
print("\n=== 2. Counts and Base Rate ===")
print(f"Total Rows: {len(df)}")
df['is_declining_label'] = df['trend_direction'] == 'down'
base_rate = df['is_declining_label'].mean()
print(f"Decline Base Rate (Label Rate): {base_rate:.2%}")
print(f"Declining Pages Count: {df['is_declining_label'].sum()}")

# 3. Missingness by Column and Category
print("\n=== 3. Missingness check ===")
missing_pct = df.isna().mean() * 100
missing_cols = missing_pct[missing_pct > 0].sort_values(ascending=False)
print("Columns with missing values (%):")
print(missing_cols)

print("\nMissingness rates (%) grouped by content_type:")
for c_type, group in df.groupby('content_type'):
    print(f"\nContent Type: {c_type} (Count: {len(group)})")
    group_missing = group.isna().mean() * 100
    print(group_missing[group_missing > 0])

# 4. Windows & avg_position = 0 Gotcha
print("\n=== 4. Windows & avg_position = 0 gotcha ===")
print("Age tiers distribution:")
print(df['age_tier'].value_counts())

zero_pos = (df['avg_position'] == 0).sum()
print(f"\nNumber of pages with avg_position == 0: {zero_pos} ({zero_pos/len(df):.2%})")
# Verify that pages with avg_position == 0 have GSC impressions >= 1
zero_pos_df = df[df['avg_position'] == 0]
print(f"Min GSC impressions for avg_position == 0 pages: {zero_pos_df['impressions_90d'].min()}")

## 4. Data limits

### Data Constraints and Caveats
1. **Systematic Missingness by Content Type**:
   - `feedly article` has **100% missing data** for keyword context (`search_volume`, `competition`, `competition_level`, `cpc`, `main_intent`) and **100% missing data** for `word_count`/`char_count`.
   - `comparison article` has **100% missing data** for keyword context, `word_count`, and `char_count`.
   - `keyword article` has **~22.8% missing data** for `word_count` and `char_count`, and a small amount of missing keyword data (~1.4%).
   - *Implication*: Using features like `search_volume` or `word_count` directly in a model without accounting for this missingness pattern will leak the `content_type` category. We must use helper indicator flags (e.g. `has_word_count`) or split models/features by `content_type`.

2. **The avg_position = 0 Gotcha**:
   - `avg_position == 0` represents **"no position data"** (1,205 rows), not page rank zero. Using it raw as a numerical feature will mislead the model since rank 0 is technically better than rank 1, but here it represents missing signal.
   - *Implication*: We must replace 0 with `NaN` or a high dummy value (e.g., 100 or 250), or use indicator features.

3. **Measurement System Discrepancies**:
   - `scroll_rate` and `ai_traffic_pct` can **exceed 100%** because the numerators and denominators come from different analytics instrumentation systems. These rates should be capped, log-transformed, or binned to handle extreme values.

4. **Window Overlap and Leakage Risk**:
   - In the starter dataset, the trend is based on `last_30d` vs `prev_30d`, meaning `last_30d` columns are locked to the label and cannot be used as features.
   - In the full warehouse release, the query table covers a fixed 90-day window. If our target is defined in a future window, we must ensure our features are aligned to the correct historical window to prevent future data from leaking into the model.

In [ ]:
# Demonstrate discrepancies and average position gotchas
print("=== Capping and extreme value check ===")
print("Max scroll_rate in raw data:", df['scroll_rate'].max())
print("Max ai_traffic_pct in raw data:", df['ai_traffic_pct'].max())

print("\n=== Check avg_position == 0 example ===")
print(df[df['avg_position'] == 0][['content_id', 'impressions_90d', 'clicks_90d', 'avg_position']].head(5))

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.